In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df=pd.read_csv('../data/processed/job_fraud_features.csv')
X=df.drop('fraudulent',axis=1)
y=df['fraudulent']

In [4]:
X.head()

,company_profile_length_log,org_count_log,gpe_count_log,company_profile_missing,required_experience_missing,info_completeness_score,has_urgency_language,has_company_logo,contract,full-time,other,part-time,temporary,unknown,has_questions,telecommuting
0,6.786717,2.302585,0.693147,0,0,11.782449,0,1,0,0,1,0,0,0,0,0
1,7.160069,1.791759,1.609438,0,0,12.561267,0,1,0,1,0,0,0,0,0,0
2,6.779922,2.197225,1.098612,0,1,11.075759,0,1,0,0,0,0,0,1,0,0
3,6.421622,1.098612,0.000000,0,0,9.520235,0,1,0,1,0,0,0,0,0,0
4,7.395722,2.944439,1.098612,0,0,13.438773,0,1,0,1,0,0,0,0,1,0


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [6]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)


fraudulent
0    0.951622
1    0.048378
Name: proportion, dtype: float64

In [7]:
from sklearn.linear_model import LogisticRegression


In [8]:
model=LogisticRegression(max_iter=1000, class_weight='balanced',random_state=42)
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

In [9]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test,y_pred)

array([[2794,  609],
       [  66,  107]])

In [10]:
y_prob=model.predict_proba(X_test)[:,1]
y_pred_04=(y_prob>=0.4).astype(int)

In [11]:
confusion_matrix(y_test,y_pred_04)

array([[2356, 1047],
       [  58,  115]])

In [12]:
df_missed=X_test[(y_pred_04==0) & (y_test==1)]
df_caught=X_test[(y_pred_04==1) & (y_test==1)]

print(df_missed[['info_completeness_score','org_count_log','company_profile_missing']].mean())
print(df_caught[['info_completeness_score','org_count_log','company_profile_missing']].mean())

info_completeness_score    9.352249
org_count_log              0.990814
company_profile_missing    0.000000
dtype: float64
info_completeness_score    1.396561
org_count_log              0.187066
company_profile_missing    0.895652
dtype: float64


In [13]:
X_test

,company_profile_length_log,org_count_log,gpe_count_log,company_profile_missing,required_experience_missing,info_completeness_score,has_urgency_language,has_company_logo,contract,full-time,other,part-time,temporary,unknown,has_questions,telecommuting
16995,0.000000,0.000000,0.000000,1,1,0.000000,0,1,0,1,0,0,0,0,0,0
9357,5.846439,0.693147,0.000000,0,0,8.539586,0,1,0,1,0,0,0,0,1,0
11561,6.628041,0.693147,1.791759,0,0,11.112948,0,1,1,0,0,0,0,0,0,0
1105,7.892452,1.791759,0.000000,0,0,11.684212,0,1,0,1,0,0,0,0,1,0
1980,6.779922,2.197225,1.098612,0,0,12.075759,0,1,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7039,0.000000,0.000000,0.000000,1,1,0.000000,0,0,0,0,0,0,0,1,0,0
14472,6.765039,2.079442,0.000000,0,1,9.844481,0,1,0,0,0,0,0,1,0,0
14453,6.511745,1.791759,0.693147,0,1,9.996652,0,1,0,1,0,0,0,0,1,0
6296,6.732211,2.079442,1.098612,0,1,10.910265,0,1,0,0,0,0,0,1,1,0


In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from scipy.sparse import hstack

df_num = pd.read_csv("../data/processed/job_fraud_features.csv")


df_raw = pd.read_csv("../data/raw/fake_job_postings.csv")


y = df_num['fraudulent']


X_num_train, X_num_test, y_train, y_test, X_raw_train, X_raw_test = train_test_split(
    df_num.drop(columns=['fraudulent']),
    y,
    df_raw,
    test_size=0.2,
    stratify=y,
    random_state=42
)

tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    min_df=5,
    ngram_range=(1, 1)
)

X_train_text = tfidf.fit_transform(X_raw_train['description'].fillna(""))
X_test_text  = tfidf.transform(X_raw_test['description'].fillna(""))

X_train_struct = X_num_train.astype(float).values
X_test_struct  = X_num_test.astype(float).values

X_train = hstack([X_train_text, X_train_struct])
X_test  = hstack([X_test_text, X_test_struct])



In [15]:
model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
confusion_matrix(y_test, y_pred)

array([[3233,  170],
       [  23,  150]])

In [16]:
y_test.value_counts()

fraudulent
0    3403
1     173
Name: count, dtype: int64

In [17]:
from sklearn.svm import LinearSVC

svm_model=LinearSVC(class_weight='balanced',random_state=42,max_iter=5000)

svm_model.fit(X_train,y_train)

y_pred_svm=svm_model.predict(X_test)

confusion_matrix(y_test,y_pred_svm)


array([[3342,   61],
       [  36,  137]])

In [18]:
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix

# Conservative XGBoost (no heavy tuning)
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

confusion_matrix(y_test, y_pred_xgb)

array([[3287,  116],
       [  32,  141]])

In [19]:
X_train

<COOrdinate sparse matrix of dtype 'float64'
	with 1198779 stored elements and shape (14304, 10016)>

In [20]:
n_tfidf=X_train_text.shape[1]

feature_names=tfidf.get_feature_names_out()



In [21]:
svm_coef=svm_model.coef_[0]

tfidf_coef=svm_coef[:n_tfidf]

coef_df=pd.DataFrame({
    'word':feature_names,
    'weight':tfidf_coef
})

top_fraud_words=coef_df.sort_values(by='weight',ascending=False).head(20)

top_real_words=coef_df.sort_values(by='weight',ascending=True).head(20)

top_fraud_words,top_real_words

(                                                   word    weight
 15                                                 1000  2.464620
 5493                                              mateo  2.329112
 9422  url_ddb080358fa5eecf5a67c649cfb4ffc343c484389f...  2.258920
 5759                                              money  2.145204
 5217                                               link  2.142582
 8994                                            timejob  2.019503
 4312                                              homes  1.849598
 1812                                           combines  1.830855
 6267                                             outlet  1.829887
 9720                                            weather  1.706764
 9668                                              wages  1.702057
 5827                                          multitask  1.695614
 8697                                           surgical  1.689740
 8608                                             subsea  1.67

In [25]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(svm_model, "models/svm_model.pkl")
joblib.dump(tfidf, "models/tfidf_vectorizer.pkl")

['models/tfidf_vectorizer.pkl']